In [2]:
import random
import math

In [3]:
import numpy as np

def init_cma_es(n):
    # --- THAM SỐ QUẦN THỂ (Slide Trang 5) ---
    lam = 4 + int(3 * np.log(n))      # lambda: pop_size
    mu = lam // 2                    # số cá thể chọn lọc

    # w_i: Trọng số (Slide Trang 6)
    weights = np.log(mu + 0.5) - np.log(np.arange(1, mu + 1))
    weights /= np.sum(weights)

    # mu_w: Số lượng cá thể hiệu quả (Slide Trang 6)
    mu_w = 1 / np.sum(weights**2)

    # --- CÁC HỆ SỐ HỌC (Slide Trang 6 & 7) ---
    c_sigma = (mu_w + 2) / (n + mu_w + 5)
    d_sigma = 1 + 2 * max(0, np.sqrt((mu_w - 1) / (n + 1)) - 1) + c_sigma # Phụ cho bước 7
    c_c = (4 + mu_w / n) / (n + 4 + 2 * mu_w / n)
    c_1 = 2 / ((n + 1.3)**2 + mu_w)
    c_mu = min(1 - c_1, 2 * (mu_w - 2 + 1/mu_w) / ((n + 2)**2 + mu_w))

    return {
        'n': n, 'lam': lam, 'mu': mu, 'weights': weights, 'mu_w': mu_w,
        'c_sigma': c_sigma, 'd_sigma': d_sigma, 'c_c': c_c, 'c_1': c_1, 'c_mu': c_mu,
        'm': np.random.uniform(-5, 5, n),
        'sigma': 0.5,
        'C': np.eye(n),
        'pc': np.zeros(n),
        'ps': np.zeros(n),
        # chiN: Kỳ vọng độ dài vector ngẫu nhiên n chiều
        'chiN': np.sqrt(n) * (1 - 1/(4*n) + 1/(21*n**2)) # Phụ cho bước 7
    }

def run_cma_es(n, iterations=100):
    p = init_cma_es(n)

    for gen in range(iterations):
        # --- BƯỚC 2: ĐỘT BIẾN (Slide Trang 6) ---
        vals, vecs = np.linalg.eigh(p['C'])
        B = vecs
        D = np.diag(np.sqrt(np.maximum(0, vals)))
        BD = B @ D

        z = np.random.randn(p['lam'], n)
        y = z @ BD.T
        x = p['m'] + p['sigma'] * y # x_i = m + sigma * y_i

        # --- BƯỚC 3 & 4: ĐÁNH GIÁ & CHỌN LỌC ---
        fitness = np.array([np.sum(ind**2) for ind in x]) # Ví dụ hàm Sphere
        idx = np.argsort(fitness)
        x_sel = x[idx[:p['mu']]]
        y_sel = y[idx[:p['mu']]]

        # --- BƯỚC 5: CẬP NHẬT TÂM m (Slide Trang 6) ---
        # x_w chính là trung bình có trọng số của các cá thể tốt nhất
        x_w = np.sum(x_sel * p['weights'][:, np.newaxis], axis=0)
        # y_w là trung bình có trọng số của các nhiễu y tương ứng
        y_w = np.sum(y_sel * p['weights'][:, np.newaxis], axis=0)

        p['m'] = x_w # m_mới = x_w (Đúng định nghĩa slide)

        # --- BƯỚC 6: CẬP NHẬT p_sigma (Slide Trang 6) ---
        C_inv_half = B @ np.diag(1.0 / np.sqrt(np.maximum(1e-10, vals))) @ B.T
        p['ps'] = (1 - p['c_sigma']) * p['ps'] + \
                  np.sqrt(p['c_sigma'] * (2 - p['c_sigma']) * p['mu_w']) * (C_inv_half @ y_w)

        # --- BƯỚC 7: CẬP NHẬT sigma (Slide Trang 6) ---
        p['sigma'] *= np.exp((p['c_sigma'] / p['d_sigma']) * (np.linalg.norm(p['ps']) / p['chiN'] - 1))

        # --- BƯỚC 8: CẬP NHẬT p_c (Slide Trang 7) ---
        p['pc'] = (1 - p['c_c']) * p['pc'] + \
                  np.sqrt(p['c_c'] * (2 - p['c_c']) * p['mu_w']) * y_w

        # --- BƯỚC 9: CẬP NHẬT MA TRẬN C (Slide Trang 7) ---
        # C = (1-c1-cmu)*C + c1*pc*pc^T + cmu*SUM(wi*yi*yi^T)
        term_rank1 = p['c_1'] * np.outer(p['pc'], p['pc'])

        term_rank_mu = np.zeros((n, n))
        for i in range(p['mu']):
            term_rank_mu += p['weights'][i] * np.outer(y_sel[i], y_sel[i])
        term_rank_mu *= p['c_mu']

        p['C'] = (1 - p['c_1'] - p['c_mu']) * p['C'] + term_rank1 + term_rank_mu

        if gen % 10 == 0:
            print(f"Gen {gen}: Fitness = {np.min(fitness):.5f}")

    return p['m']

best = run_cma_es(n=2)

Gen 0: Fitness = 1.37224
Gen 10: Fitness = 0.03292
Gen 20: Fitness = 0.00010
Gen 30: Fitness = 0.00000
Gen 40: Fitness = 0.00000
Gen 50: Fitness = 0.00000
Gen 60: Fitness = 0.00000
Gen 70: Fitness = 0.00000
Gen 80: Fitness = 0.00000
Gen 90: Fitness = 0.00000
